<a href="https://colab.research.google.com/github/Oldmanne13/face/blob/main/9face_med_dropout_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Forberedelse og Forbindelse til google drev -  Tjek GPU

In [1]:
import tensorflow as tf
import os
from google.colab import drive

print("--- Trin 1: Forberedelse ---")

# Tjek GPU
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    print(" T4 GPU ikke fundet! Tjek Runtime-indstillinger.")
else:
    print("Succes! T4 GPU er aktiv og klar.")

# Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("Google Drive er monteret.")
else:
    print("Google Drive var allerede monteret.")

--- Trin 1: Forberedelse ---
Succes! T4 GPU er aktiv og klar.
Mounted at /content/drive
Google Drive er monteret.


Udpak data (archive (6).zip) til  /content/local_data

In [2]:
print("--- Trin 2: Udpakning af billeder ---")

zip_path = '/content/drive/MyDrive/ansigter.zip'
output_folder = '/content/local_data'

if os.path.exists(zip_path):
    print("Pakker zip-fil ud...(dette kan tage mange minutter).")
    !unzip -q "{zip_path}" -d "{output_folder}"
    print(f" Udpakning færdig! Filerne ligger nu i: {output_folder}")
else:
    print(" FEJL: Kunne ikke finde zip-filen. Tjek stien.")

--- Trin 2: Udpakning af billeder ---
 FEJL: Kunne ikke finde zip-filen. Tjek stien.


Data ligger forebredt til YOLO (You Only Look Once) algoritme til objektgenkendelse, men da det er tensorflow værktøjskassen der er vist på kursus. Data ligger fordelt i 9 facial expressions you need/ test, 9..../train og 9...../valid. data.aml indeholder stier til data og klasser (udtryk)

In [3]:
import os
import shutil
import yaml

print("--- Trin: Omorganiserer YOLO til TensorFlow format ---")

# 1. Hent klassenavnene fra din data.yaml
yaml_path = '/content/local_data/9 Facial Expressions you need/data.yaml'
with open(yaml_path, 'r') as f:
    data_info = yaml.safe_load(f)
    class_names = data_info['names']

print(f"Fundet 9 udtryk: {class_names}")

# 2. Funktion til at sortere billeder
def sort_yolo_to_folders(split):
    base = f'/content/local_data/9 Facial Expressions you need/{split}'
    img_dir = os.path.join(base, 'images')
    lbl_dir = os.path.join(base, 'labels')
    target_dir = f'/content/tf_data/{split}'

    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith('.txt'): continue

        # Læs første tal i txt-filen (klassen)
        with open(os.path.join(lbl_dir, lbl_file), 'r') as f:
            first_line = f.readline().split()
            if not first_line: continue
            class_id = int(first_line[0])
            emotion_name = class_names[class_id]

        # Lav ny mappe: /content/tf_data/train/Happy/
        new_folder = os.path.join(target_dir, emotion_name)
        os.makedirs(new_folder, exist_ok=True)

        # Flyt billedet
        img_name = lbl_file.replace('.txt', '.jpg') # Eller .png
        src_img = os.path.join(img_dir, img_name)
        if os.path.exists(src_img):
            shutil.copy(src_img, os.path.join(new_folder, img_name))

# 3. Kør sorteringen for både train og valid
sort_yolo_to_folders('train')
sort_yolo_to_folders('valid')
print("✅ Færdig! Dine data er nu sorteret til TensorFlow.")

--- Trin: Omorganiserer YOLO til TensorFlow format ---


FileNotFoundError: [Errno 2] No such file or directory: '/content/local_data/9 Facial Expressions you need/data.yaml'

Indlæsning af Datasæt

In [ ]:
print("Indlæser datasæt til TensorFlow ")

# base_path = '/content/local_data/9 Facial Expressions you need'
# Din nye sti efter sorteringen
base_path = '/content/tf_data'

# Kør herefter din image_dataset_from_directory igen
# Nu vil den finde 9 klasser i stedet for 2!
# 1. Indlæs de rå datasæt først
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base_path, 'train'),
    image_size=(64, 64),
    batch_size=32,
    label_mode='categorical'
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base_path, 'valid'),
    image_size=(64, 64),
    batch_size=32,
    label_mode='categorical'
)

# 2. GEM klassenavnene nu, mens vi har adgang til dem!
class_names = raw_train_ds.class_names
print(f"✅ Klassenavne gemt: {class_names}")

# 3. Optimering (Her skifter objektet type til '_PrefetchDataset')
AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = raw_val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("✅ Datasæt er optimeret og klar til T4 GPU.")

Design af AI-Modellen (CNN)
Dette er hjernen i projektet. Vi bruger et Convolutional Neural Network (CNN), som er standard i undervisningen til billeder.

In [ ]:
import os
# Vi tæller hvor mange undermapper der er i din trænings-mappe
antal_klasser = len(os.listdir('/content/tf_data/train'))

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 3)), # billedet er 64x64 pixel og rgb

    # Data Augmentation (Indbygget i modellen) laver tilfældig rotation
    # og flipper billede - giver mere data at træne på.
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.Rescaling(1./255), # Normalisering skalere 0-255 til 0-1

    # Blok 1
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'), # bruge 32 filtre, med kernel på 3x3 pixels for at scanne billede.
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2, 2),# Gør data mindre og fjerner unødvendig detaljer

    # Blok 2
    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),# billedet er blevet mindre så kan flere filtre bruges
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Dropout(0.2), # Smid 20% af forbindelserne væk her for at" tvinge" nadre neuroner til at arbejde/træne

    # Klassificering
    tf.keras.layers.Flatten(),# 2d til 1d
    tf.keras.layers.Dense(128, activation='relu'), # Hvad forestiller billedet
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5), # Kraftig dropout før finalen for at sikre at modellen ikke lærer billeder udenad
    tf.keras.layers.Dense(antal_klasser, activation='softmax') # 9 klasser
    ])
model.compile(
    optimizer='adam', #justere selv steps
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f" Model er klar til at genkende {antal_klasser} ansigtsudtryk.")

Indlærings rate kan justeres ved at importer adam selv og definere indlæringsrate.
Juster Dropout hvis der ses overfitting
Batch Size	32 eller 64 som standard.
Juster antal filtre
from tensorflow.keras.optimizers import Adam
Standard er ofte 0.001.
Prøv 0.0001 hvis modellen er ustabil, eller 0.01 hvis den lærer for langsomt.
min_optimizer = Adam(learning_rate=0.0001)
model.compile(
    optimizer=min_optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Træning med 50 epocs

In [ ]:
print("Starter træning ---")

# Vi starter med 50 runder (epochs)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50
)

print("TRÆNING FÆRDIG!")

In [ ]:
import matplotlib.pyplot as plt

print("Evaluerer træning med grafer ---")

# Lav en figur med to grafer ved siden af hinanden
plt.figure(figsize=(12, 4))

# Graf 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Træning')
plt.plot(history.history['val_accuracy'], label='Validering')
plt.title('Modellens Nøjagtighed (Accuracy)')
plt.xlabel('Epochs')
plt.ylabel('Nøjagtighed')
plt.legend()

# Graf 2: Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Træning')
plt.plot(history.history['val_loss'], label='Validering')
plt.title('Modellens Fejlrate (Loss)')
plt.xlabel('Epochs')
plt.ylabel('Fejl')
plt.legend()

plt.show()

Modellen bruger 50 % dropout hvilken giver anledning til de meget store spikes, der observeres samtidig performer modellen bedre under validering end under træning. Det ses også at modellen hurtigt forbedres men flader ud. Dette kan skyldes at der learningsraten er adaptiv ved brug af "adam". Næste trin kunne være at sænke dropout og arbejde med learningrate.

In [ ]:
import numpy as np

print("Test på et ukendt billede ---")

# Vi tager et batch fra test-sættet (hvis du har defineret et, ellers brug val_ds)
# Her bruger vi val_ds som eksempel
for images, labels in val_ds.take(6):
    val_ds = val_ds.shuffle(buffer_size=1000) # sørger for, at datasættet bliver blandet
    # Vælg det første billede i batchet
    img = images[17].numpy().astype("uint8")
    true_label = class_names[np.argmax(labels[0])]

    # Lad modellen gætte
    # Vi skal tilføje en ekstra dimension (batch dimension) med expand_dims
    prediction = model.predict(np.expand_dims(images[0], axis=0))
    predicted_label = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100

    # Vis billedet og resultatet
    plt.imshow(img)
    plt.title(f"Gæt: {predicted_label} ({confidence:.1f}%)\nSandhed: {true_label}")
    plt.axis("off")
    plt.show()